# MEIDNet on your own data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ABnano/MEIDNet/blob/main/notebooks/02_your_own_data.ipynb)

Bring a **table** (CSV or Excel) with one row per material: an id column, one or more numeric property columns, and the crystal structure of each row, either as CIF text in a column or as a zip of `<id>.cif` files. MEIDNet checks the data, trains a model on it and designs new materials for your targets, block by block as in MEIDNet Studio.

No data at hand? Tick **use_example** in the first form and 600 rows of Perov-5 are used instead. Training is faster on a GPU (*Runtime → Change runtime type → GPU*), but a CPU is fine for a first try with a few epochs.

<p><span style="display:inline-block;border-left:5px solid #2a78d6;background:rgba(42,120,214,0.09);border-radius:4px;padding:2px 9px;margin:2px 0">📊 1 · Data</span> → <span style="display:inline-block;border-left:5px solid #eb6834;background:rgba(235,104,52,0.09);border-radius:4px;padding:2px 9px;margin:2px 0">🧠 2 · Model</span> → <span style="display:inline-block;border-left:5px solid #1baf7a;background:rgba(27,175,122,0.09);border-radius:4px;padding:2px 9px;margin:2px 0">🧱 3 · Family</span> → <span style="display:inline-block;border-left:5px solid #eda100;background:rgba(237,161,0,0.09);border-radius:4px;padding:2px 9px;margin:2px 0">📏 4 · Rules</span> → <span style="display:inline-block;border-left:5px solid #e87ba4;background:rgba(232,123,164,0.09);border-radius:4px;padding:2px 9px;margin:2px 0">🎯 5 · Targets</span> → <span style="display:inline-block;border-left:5px solid #008300;background:rgba(0,131,0,0.09);border-radius:4px;padding:2px 9px;margin:2px 0">🔎 6 · Search</span> → <span style="display:inline-block;border-left:5px solid #4a3aa7;background:rgba(74,58,167,0.09);border-radius:4px;padding:2px 9px;margin:2px 0">💎 7 · Candidates</span></p>

**How to use it:** run the cells from top to bottom (Shift+Enter, or *Runtime → Run all*). Cells with fields are forms: change a value, then run that cell and the ones below it again.

<sub>Notebook by Anand Babu. MEIDNet: <i>Multimodal generative AI framework for inverse materials design</i>, npj Computational Materials (2026), <a href="https://doi.org/10.1038/s41524-026-02153-3">doi:10.1038/s41524-026-02153-3</a>. 3D views use chemiscope: G. Fraux, R. K. Cersonsky, M. Ceriotti, <i>Chemiscope: interactive structure-property explorer for materials and molecules</i>, JOSS 5, 2117 (2020).</sub>

In [ ]:
#@title ⚙️ Setup: install MEIDNet and load the helpers (run this cell first)
#@markdown In Colab this installs MEIDNet (PyTorch for CPU is enough), chemiscope and ASE, then defines the helpers that every block uses: charts in the block colours, 3D views and report display. It takes about a minute the first time.
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install torch --index-url https://download.pytorch.org/whl/cpu
    !pip -q install git+https://github.com/ABnano/MEIDNet.git chemiscope ase
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()       # chemiscope's 3D viewer is a notebook widget
    from importlib.metadata import version as _installed
    if "numpy" in sys.modules and sys.modules["numpy"].__version__ != _installed("numpy"):
        print("pip updated numpy: choose Runtime -> Restart session, then run this cell again.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
try:
    import meidnet
except ImportError:
    raise ImportError("MEIDNet is not installed in this Python: "
                      "pip install git+https://github.com/ABnano/MEIDNet.git chemiscope ase") from None

# one colour per block, as in MEIDNet Studio; ink and paper of the charts
BLOCK = {"data": "#2a78d6", "model": "#eb6834", "family": "#1baf7a", "rules": "#eda100",
         "targets": "#e87ba4", "search": "#008300", "candidates": "#4a3aa7"}
INK, INK2, MUTED, GRID, AXIS, PAPER, REST = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb", "#d3d2cb"
for _key, _value in {
        "figure.figsize": (6.4, 3.6), "figure.dpi": 110, "figure.facecolor": PAPER, "savefig.facecolor": PAPER,
        "axes.facecolor": PAPER, "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.spines.top": False,
        "axes.spines.right": False, "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID,
        "grid.linewidth": 0.8, "grid.linestyle": "-", "axes.titlesize": 12, "axes.titleweight": "bold",
        "axes.titlelocation": "left", "axes.titlecolor": INK, "axes.labelcolor": INK2, "axes.labelsize": 10,
        "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
        "legend.frameon": False, "legend.fontsize": 9, "lines.linewidth": 2,
        "lines.solid_capstyle": "round"}.items():
    try:
        plt.rcParams[_key] = _value
    except (KeyError, ValueError):                    # an older matplotlib: keep its default
        pass


def print_flow(to, *items):
    """The last line of a block: what it hands on to the next block."""
    print(f"-> {to}: " + "; ".join(str(i) for i in items))


# ── charts: one axis each, block colours for the data, plain-word titles and labels ──
def _finish(fig, ax, title, xlabel, ylabel, subtitle=None):
    ax.set_title(title, pad=22 if subtitle else 8)
    if subtitle:
        ax.text(0, 1.02, subtitle, transform=ax.transAxes, ha="left", va="bottom", fontsize=9, color=INK2)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    fig.tight_layout()
    plt.show()


def _window_text(lo, hi):
    if lo is not None and hi is not None:
        return f"{lo:g} to {hi:g}"
    return f"at least {lo:g}" if lo is not None else f"at most {hi:g}"


def hist_prop(values, label, unit="", colour=BLOCK["data"], target=None, window=None, bins=40, title=None,
              ylabel="count"):
    """Histogram in a block colour, with an optional target line and allowed window (min, max)."""
    v = pd.to_numeric(pd.Series(list(values), dtype="object"), errors="coerce").astype(float)
    v = v[np.isfinite(v)]
    if v.empty:
        print(f"(no values to plot for {label})")
        return
    fig, ax = plt.subplots()
    ax.hist(v, bins=bins, color=colour, edgecolor=PAPER, linewidth=0.8)
    if window is not None:
        lo, hi = window
        x0, x1 = ax.get_xlim()
        a, b = (x0 if lo is None else lo), (x1 if hi is None else hi)
        ax.axvspan(a, b, color=colour, alpha=0.14, lw=0, zorder=0, label=f"allowed: {_window_text(lo, hi)}")
        ax.set_xlim(min(x0, a), max(x1, b))
    if target is not None:
        ax.axvline(target, color=INK, lw=1.5, label=f"target: {target:g} {unit}".strip())
    if window is not None or target is not None:
        ax.legend(loc="best")
    unit_text = f" [{unit}]" if unit else ""
    stats = f"{len(v):,} values, median {np.median(v):.3g}{' ' + unit if unit else ''}"
    _finish(fig, ax, title or label, f"{label}{unit_text}", ylabel, subtitle=stats)


def funnel_plot(stages, colour=BLOCK["rules"], title="Compositions still allowed after each rule"):
    """Horizontal bars: how many compositions survive each rule, applied in order (stages: [(label, count)])."""
    labels, counts = [str(s[0]) for s in stages], [int(s[1]) for s in stages]
    fig, ax = plt.subplots(figsize=(6.4, 0.42 * len(stages) + 1.2))
    y = np.arange(len(stages))[::-1]
    ax.barh(y, counts, height=0.6, color=colour, edgecolor=PAPER, linewidth=0.8)
    top = max(counts) or 1
    for yi, c in zip(y, counts):
        ax.text(c + 0.01 * top, yi, f"{c:,}", va="center", ha="left", color=INK2, fontsize=9)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlim(0, top * 1.15)
    ax.grid(axis="y", visible=False)
    _finish(fig, ax, title, "compositions still allowed", "")


def parity_plot(true, pred, label, unit="", colour=BLOCK["model"]):
    """Predicted against true values; points on the grey diagonal are perfect predictions."""
    t, p = np.asarray(true, dtype=float), np.asarray(pred, dtype=float)
    mae = float(np.mean(np.abs(p - t)))
    ss = float(np.sum((t - t.mean()) ** 2))
    r2 = 1 - float(np.sum((p - t) ** 2)) / ss if ss > 0 else float("nan")
    lo, hi = float(min(t.min(), p.min())), float(max(t.max(), p.max()))
    pad = 0.05 * ((hi - lo) or 1.0)
    fig, ax = plt.subplots(figsize=(4.8, 4.6))
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color=AXIS, lw=1, zorder=1)
    ax.scatter(t, p, s=28, color=colour, edgecolor=PAPER, linewidth=0.8, zorder=2)
    ax.set_xlim(lo - pad, hi + pad)
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_aspect("equal", adjustable="box")
    u = f" [{unit}]" if unit else ""
    _finish(fig, ax, f"{label}: predicted vs true", f"true {label}{u}", f"predicted {label}{u}",
            subtitle=f"mean error {mae:.3g}{' ' + unit if unit else ''}, R² {r2:.2f}, {len(t):,} materials")


def loss_curves(ckpt, colour=BLOCK["model"]):
    """Training loss and validation error per epoch, from the history saved in a MEIDNet checkpoint."""
    import torch
    saved = torch.load(ckpt, map_location="cpu", weights_only=False)
    history = saved.get("history") if isinstance(saved, dict) else None
    if not history or not history.get("train"):
        print("this checkpoint stores no training history (the published v1 model does not)")
        return
    from matplotlib.ticker import MaxNLocator
    train_log, val_log = history["train"], history.get("val") or []
    dots = dict(marker="o", ms=5, mec=PAPER, mew=1)
    losses = [r["total"] for r in train_log]
    fig, ax = plt.subplots()
    ax.plot([r["epoch"] for r in train_log], losses, color=colour, **(dots if len(train_log) <= 30 else {}))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    wide = min(losses) > 0 and max(losses) / min(losses) > 20          # a log axis only when it helps
    if wide:
        ax.set_yscale("log")
    _finish(fig, ax, "Training loss (lower is better)", "epoch", "loss (log scale)" if wide else "loss",
            subtitle=f"{len(train_log)} epochs, last value {losses[-1]:.3g}")
    units = {p["column"]: p.get("unit", "") for p in saved.get("properties", [])}
    for prop in (val_log[-1]["mae"] if val_log else {}):
        fig, ax = plt.subplots()
        ax.plot([r["epoch"] for r in val_log], [r["mae"][prop] for r in val_log], color=colour,
                **(dots if len(val_log) <= 30 else {}))
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        u = units.get(prop, "")
        _finish(fig, ax, f"Validation error of {prop} (lower is better)", "epoch",
                f"mean absolute error{' [' + u + ']' if u else ''}",
                subtitle=f"last value {val_log[-1]['mae'][prop]:.3g}{' ' + u if u else ''}")


def space_scatter(table, x, y, colour=BLOCK["targets"], target=None, highlight=None,
                  highlight_label="closest to the target", xlabel=None, ylabel=None,
                  title="Every composition, as the model predicts it"):
    """Compositions that pass every rule in the block colour, the rejected ones in grey, the target as a cross."""
    ok = table["passes_all"].astype(bool)
    fig, ax = plt.subplots(figsize=(7.8, 4.4))
    ax.scatter(table.loc[~ok, x], table.loc[~ok, y], s=12, color=REST, lw=0,
               label=f"rejected by a rule ({int((~ok).sum()):,})")
    ax.scatter(table.loc[ok, x], table.loc[ok, y], s=30, color=colour, edgecolor=PAPER, lw=0.8,
               label=f"passes every rule ({int(ok.sum()):,})")
    if highlight is not None and len(highlight):
        ax.scatter(highlight[x], highlight[y], s=90, facecolor="none", edgecolor=INK, lw=1.2,
                   label=f"{highlight_label} ({len(highlight)})")
        first = highlight.iloc[0]
        ax.annotate(first["formula"], (first[x], first[y]), xytext=(7, 7), textcoords="offset points",
                    fontsize=9, color=INK)
    if target is not None:
        tx, ty = target
        if ty is None:
            ax.axvline(tx, color=INK, lw=1.5, label="target")
        else:
            ax.scatter([tx], [ty], marker="+", s=220, color=INK, lw=2, zorder=5, label="target")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0)
    _finish(fig, ax, title, xlabel or x, ylabel or y)


# ── 3D: chemiscope (G. Fraux, R. K. Cersonsky, M. Ceriotti, JOSS 5, 2117, 2020) ──
def _as_structure(item):
    """A pymatgen Structure from a Structure, a chemiscope structure dict or an ase.Atoms."""
    from pymatgen.core import Lattice, Structure
    if isinstance(item, Structure):
        return item
    if isinstance(item, dict):
        xyz = np.column_stack([item["x"], item["y"], item["z"]])
        return Structure(Lattice(np.reshape(item["cell"], (3, 3))), item["names"], xyz, coords_are_cartesian=True)
    from pymatgen.io.ase import AseAtomsAdaptor
    return AseAtomsAdaptor.get_structure(item)


def _num(x):
    try:
        f = float(x)
    except (TypeError, ValueError):
        return 0.0
    return f if np.isfinite(f) else 0.0


def _chemiscope_property(name, value, n):
    prop = dict(value) if isinstance(value, dict) else {"values": value}
    values = list(prop["values"])
    if len(values) != n:
        raise ValueError(f"property '{name}' has {len(values)} values for {n} structures")
    if not all(isinstance(x, str) for x in values):
        values = [_num(x) for x in values]          # one type per property, finite numbers only
    out = {"target": prop.get("target", "structure"), "values": values}
    for key in ("units", "description"):
        if prop.get(key):
            out[key] = str(prop[key])
    return out


def show3d(structures, properties=None, settings=None):
    """
    Structures in 3D next to a map of their properties, with chemiscope (G. Fraux, R. K. Cersonsky,
    M. Ceriotti, JOSS 5, 2117, 2020). Click a point of the map to see its crystal. Without chemiscope
    a small table (formula, a, space group) is printed instead.
    """
    structs = [_as_structure(s) for s in structures]
    if not structs:
        print("(no structures to show)")
        return
    props = {name: _chemiscope_property(name, v, len(structs)) for name, v in (properties or {}).items()}
    if sum(not isinstance(p["values"][0], str) for p in props.values()) < 2:     # the map needs two numbers
        props.setdefault("a", {"target": "structure", "values": [round(s.lattice.a, 4) for s in structs],
                               "units": "Å", "description": "cell edge a"})
        props.setdefault("volume_per_atom", {"target": "structure", "units": "Å^3",
                                             "values": [round(s.volume / len(s), 4) for s in structs],
                                             "description": "cell volume per atom"})
    try:
        import chemiscope
        from pymatgen.io.ase import AseAtomsAdaptor
        atoms = [AseAtomsAdaptor.get_atoms(s) for s in structs]
        display(chemiscope.show(atoms, properties=props, settings=settings))
    except Exception as err:                          # no chemiscope (or no widget support): a small table
        print(f"(3D view not available here: {type(err).__name__}: {err})")
        rows = []
        for s in structs[:12]:
            try:
                group = s.get_space_group_info(symprec=0.1)[0]
            except Exception:
                group = "?"
            rows.append({"formula": s.composition.reduced_formula, "a (Å)": round(s.lattice.a, 3),
                         "space group": group})
        print(pd.DataFrame(rows).to_string(index=False))
        if len(structs) > 12:
            print(f"... and {len(structs) - 12} more")


def show_dataset(dataset):
    """Show a chemiscope dataset made by MEIDNet (meidnet.studio.chemiscope or Studio.chemiscope) with show3d."""
    if not dataset.get("structures"):
        print(dataset.get("note", "nothing to show yet"))
        return
    print(f"{dataset['meta']['name']}: {dataset['meta']['description']}")
    show3d(dataset["structures"], dataset.get("properties"), dataset.get("settings"))


# ── tables ──
def space_table(space):
    """meidnet.designspace.enumerate_space(...) as a table: one row per composition, a column per rule."""
    rules = [r["name"] for r in space["rules"]]
    rows = []
    for r in space["rows"]:
        row = {"formula": r["f"], **{f"site_{g}": e for g, e in r["e"].items()}, "a": r["a"]}
        for name in rules:
            row[name] = r["d"].get(name)
            row[f"ok_{name}"] = bool(r["ok"].get(name, True))
        row["passes_all"] = bool(all(r["ok"].values()))
        row.update({f"pred_{c}": v for c, v in r["p"].items()})
        rows.append(row)
    return pd.DataFrame(rows)


def rule_window(params):
    """The allowed (min, max) of a rule from its parameters, or None for a pass/fail rule."""
    lo, hi = params.get("min", params.get("low")), params.get("max", params.get("high"))
    return None if lo is None and hi is None else (lo, hi)


def measured(rule):
    """(what a rule measures, unit), in plain words, for axis labels; other rules fall back to their title."""
    p = rule["params"]
    known = {"min_distance": ("closest distance between two atoms", "Å"),
             "bond_window": (f"nearest {p.get('to', 'B')}–{p.get('from', 'X')} distance ÷ sum of ionic radii", ""),
             "tolerance_factor": ("tolerance factor t", ""),
             "octahedral_factor": ("octahedral factor μ = r_B / r_X", "")}
    return known.get(rule.get("rule", rule["name"]), (rule["title"], ""))


def funnel_stages(table, rules):
    """[(label, count)]: compositions still allowed after each rule, in the family's order."""
    alive = pd.Series(True, index=table.index)
    stages = [("all compositions", len(table))]
    for rule in rules:
        alive &= table[f"ok_{rule['name']}"].astype(bool)
        stages.append((rule["title"], int(alive.sum())))
    return stages


def rank_by_target(table, target, objectives):
    """Sort compositions by the score the search ranks its candidates with (0 = exactly on target)."""
    from meidnet.generate import objective_distance
    score = np.zeros(len(table))
    for o in objectives:
        o = o if isinstance(o, dict) else o.model_dump()
        loss = o.get("select_loss") or o.get("loss", "l2")
        score += o.get("select_weight", 1.0) * np.array(
            [objective_distance(float(v), float(target[o["property"]]), loss) for v in table[f"pred_{o['property']}"]])
    return table.assign(score=score).sort_values("score")


def candidates_table(candidates):
    """One row per saved candidate: where it was found, its predictions and every rule (✓ + measured value)."""
    rows = []
    for c in candidates:
        c = c if isinstance(c, dict) else c.to_dict()
        row = {"formula": c["formula"], "target": c.get("target_index", 1), "round": c["round"],
               "a (Å)": round(c["lattice_a"], 3)}
        row.update({f"predicted {k}": round(v, 3) for k, v in c["predictions"].items()})
        for r in c["constraint_results"]:
            value = r.get("value")
            shown = f" {value:.3g}" if value is not None and r["name"] != "charge_neutrality" else ""
            row[r["name"]] = ("✓" if r["passed"] else "✗") + shown
        row["warnings"] = "; ".join(c.get("flags") or [])
        rows.append(row)
    return pd.DataFrame(rows)


# ── model helpers ──
def predict_records(lm, records):
    """True and predicted (structure -> properties) values for records read by meidnet.pipeline.check."""
    import torch
    x = torch.tensor(np.stack([r.dense for r in records]), dtype=torch.float32, device=lm.device)
    with torch.no_grad():
        z, _ = lm.model.encode_crystal(x)
        predicted = lm.stats.denormalize_tensor(lm.model.property_decoder(z)).cpu().numpy()
    return np.stack([r.properties for r in records]), predicted


def validation_records(cfg):
    """The materials that training held out (the same split as `meidnet train`), for an honest parity plot."""
    from meidnet.data import split_records
    from meidnet.pipeline import check
    info = check(cfg, write_report=False)
    held_out = info["val_records"] or split_records(info["records"], cfg.data.val_fraction, cfg.training.seed)[1]
    if not held_out:
        print("too few materials for a validation split: showing the training materials instead")
        held_out = info["records"]
    return held_out


def show_report(path, height=900):
    """Show one of MEIDNet's HTML reports (check, training, generation) inside the notebook."""
    import html as _html
    with open(path, encoding="utf-8") as f:
        page = f.read()
    if IN_COLAB:                                      # Colab gives every output its own frame
        display(HTML(page))
    else:                                             # elsewhere: a frame keeps the report's style to itself
        display(HTML(f'<iframe srcdoc="{_html.escape(page)}" style="width:100%;height:{height}px;'
                     f'border:1px solid {GRID};border-radius:8px"></iframe>'))
    print("report file:", os.path.abspath(path))


print(f"MEIDNet {meidnet.__version__} is ready" + (" in Colab" if IN_COLAB else ""))

<div style="border-left:6px solid #2a78d6;background:rgba(42,120,214,0.09);border-radius:6px;padding:12px 18px;margin:6px 0">
<h2 style="margin:0 0 6px 0">📊 Block 1 · Data</h2>
<p style="margin:0">What MEIDNet learns from: a table of materials with their structures and properties.</p>
</div>

MEIDNet reads every row exactly as training will, and tells you which rows it can use and why the others were skipped. Each structure is also *aligned* to the family's prototype (atom *i* on site *i*), which generation needs.

In [ ]:
#@title Your data
use_example = False  #@param {type:"boolean"}
#@markdown Untick to upload your own table (and, if you have one, a .zip of `<id>.cif` files) when the next cell runs; tick to use 600 rows of Perov-5.

In [ ]:
import re
import zipfile
from meidnet.data import read_table
if use_example:
    !meidnet download-data
    pd.read_csv("data/perov5/train.csv").head(600).to_csv("example_materials.csv", index=False)
    table_file = "example_materials.csv"
elif IN_COLAB:
    from google.colab import files
    uploaded = files.upload()                 # choose your table, plus a .zip of CIF files if you have one
    for name in uploaded:
        if name.lower().endswith(".zip"):
            os.makedirs("structures", exist_ok=True)
            with zipfile.ZipFile(name) as z:
                for member in z.namelist():
                    if member.lower().endswith(".cif"):
                        with open(os.path.join("structures", os.path.basename(member)), "wb") as out:
                            out.write(z.read(member))
            print(len(os.listdir("structures")), "CIF files in structures/")
    table_file = next((n for n in uploaded if n.lower().endswith((".csv", ".xlsx", ".xls", ".json"))), None)
    if table_file is None:
        raise ValueError("Please upload a table (.csv or .xlsx) with one row per material.")
else:
    table_file = "materials.csv"              # outside Colab: put your table next to this notebook
df = read_table(table_file)
print(f"{table_file}: {len(df):,} rows; columns: {', '.join(map(str, df.columns))}")
display(df.head())

In [ ]:
#@title Describe your table
id_column = "material_id"        #@param {type:"string"}
cif_column = "cif"               #@param {type:"string"}
properties = "heat_all dir_gap"  #@param {type:"string"}
family = "perovskite_abx3"       #@param ["perovskite_abx3", "double_perovskite_a2bbx6"]
variant = "halide"               #@param ["halide", "oxide", "chalcogenide", "nitride"] {allow-input: true}
prototype_tolerance = 0.15       #@param {type:"number"}
#@markdown **properties**: the numeric columns to learn, separated by spaces. **cif_column**: the column with the CIF text (not used when you uploaded a zip of CIF files). **variant**: halide, oxide, chalcogenide or nitride for ABX₃; halide or oxide for A₂BB′X₆. **prototype_tolerance**: how far (as a fraction of a cell edge) an atom may sit from its ideal site; raise it to accept distorted structures.

In [ ]:
init_args = (f'--name mine --table "{table_file}" --id-column {id_column} --cif-column {cif_column or "cif"} '
             f'--properties {properties} --family {family}' + (f" --variant {variant}" if variant.strip() else ""))
!meidnet init {init_args} --force
text = open("meidnet.yaml", encoding="utf-8").read()
if os.path.isdir("structures") and not use_example:      # the structures come from the zip of CIF files
    text = re.sub(r"(?m)^  cif_column: .*$", "  cif_column: null\n  structures_dir: structures", text)
text = re.sub(r"(?m)^(  align_to_prototype: .*)$", rf"\1\n  prototype_tolerance: {prototype_tolerance}", text)
with open("meidnet.yaml", "w", encoding="utf-8") as f:
    f.write(text)

from meidnet.config import load_config
from meidnet.pipeline import check
cfg = load_config("meidnet.yaml")
info = check(cfg, keep_structures=True)       # reads every row as training will; writes check_report.html
rep, records = info["report"], info["records"]
print(f"\n{rep.kept:,} of {rep.rows:,} rows are usable ({rep.aligned:,} aligned to the {cfg.family_name} prototype)")
if rep.skipped:
    display(pd.DataFrame([{"skipped because": reason, "rows": n, "for example": ", ".join(rep.examples.get(reason, []))}
                          for reason, n in rep.skipped.most_common()]))
for j, p in enumerate(cfg.data.properties):
    hist_prop([r.properties[j] for r in records], p.display, p.unit, colour=BLOCK["data"], ylabel="materials",
              title=f"{p.display} in your usable data")

Your usable structures in 3D, after alignment to the prototype (chemiscope; up to 300 shown).

In [ ]:
shown = records[:300]
if shown:
    show3d([r.structure for r in shown],
           properties={"formula": [r.formula for r in shown], "material_id": [r.material_id for r in shown],
                       **{p.column: {"values": [float(r.properties[j]) for r in shown], "units": p.unit}
                          for j, p in enumerate(cfg.data.properties)}})
print_flow("Model", f"{rep.kept:,} usable materials with {len(cfg.data.properties)} properties "
                    f"({', '.join(p.column for p in cfg.data.properties)})")

<div style="border-left:6px solid #eb6834;background:rgba(235,104,52,0.09);border-radius:6px;padding:12px 18px;margin:6px 0">
<h2 style="margin:0 0 6px 0">🧠 Block 2 · Model</h2>
<p style="margin:0">One shared latent space where structures and properties meet.</p>
</div>

Training teaches the two encoders to meet in one latent space: the crystal of each material and its property values must land on the same point. Start with a few epochs to see the whole pipeline work, then raise *epochs* (200 is typical). The training report says in plain words how good the model is.

In [ ]:
#@title Training
epochs = 30  #@param {type:"integer"}

In [ ]:
from meidnet.pipeline import train
from meidnet.checkpoint import describe, load_checkpoint, property_ranges
ckpt = train(cfg, epochs=epochs)              # writes runs/mine/model.pt and training_report.html
lm = load_checkpoint(ckpt)
print(describe(lm))
loss_curves(ckpt)

How accurate is it? The materials held out from training (the validation split) are predicted **from their structure alone**, the same situation as judging a new candidate. Points on the grey diagonal are perfect predictions.

In [ ]:
held_out = validation_records(cfg)
true, predicted = predict_records(lm, held_out)
for j, (label, unit) in enumerate(zip(lm.stats.labels, lm.stats.units)):
    parity_plot(true[:, j], predicted[:, j], label, unit, colour=BLOCK["model"])
show_report(os.path.join(cfg.out_dir, "training_report.html"))
errors = ", ".join(f"{c} {np.mean(np.abs(predicted[:, j] - true[:, j])):.3g}" for j, c in enumerate(lm.stats.columns))
print_flow("Family", f"your model (mean validation error: {errors})")

<div style="border-left:6px solid #1baf7a;background:rgba(27,175,122,0.09);border-radius:6px;padding:12px 18px;margin:6px 0">
<h2 style="margin:0 0 6px 0">🧱 Block 3 · Family</h2>
<p style="margin:0">The crystal type: where atoms sit and which elements may occupy each site.</p>
</div>

Generation fills the prototype your structures were aligned to with new combinations of elements. The family and variant come from `meidnet.yaml` (the form of the Data block).

In [ ]:
from meidnet.pipeline import family_for
from meidnet.designspace import enumerate_space, total_compositions
from meidnet.constraints import build_candidate, evaluate
cfg.generation.overrides = {}                 # the Rules block may set a window later
fam = family_for(cfg, need_variant=True)      # the family and variant named in meidnet.yaml
print(fam.describe())
display(pd.DataFrame([{"site group": g, "atoms per cell": len(grp.slots), "elements allowed": " ".join(grp.sample)}
                      for g, grp in fam.groups.items()]))
example = evaluate(build_candidate(fam, {g: grp.sample[0] for g, grp in fam.groups.items()}), fam.constraints)
print(f"\nOne way to fill the prototype: {example.formula()}; it passes every rule: {example.passed} "
      "(the Rules block checks them all)")
show3d([example.raw], properties={"formula": [example.formula()]})
print_flow("Rules", f"{total_compositions(fam):,} compositions (every way to fill the sites)")

<div style="border-left:6px solid #eda100;background:rgba(237,161,0,0.09);border-radius:6px;padding:12px 18px;margin:6px 0">
<h2 style="margin:0 0 6px 0">📏 Block 4 · Rules</h2>
<p style="margin:0">Hard checks that every candidate must pass, in order.</p>
</div>

Every composition of the family is checked against its rules, in order, and **your** model predicts its properties. Each rule measures one number and compares it with an allowed window.

In [ ]:
space = enumerate_space(fam, lm)              # every composition: the value of each rule + predicted properties
table = space_table(space)
for rule in space["rules"]:
    print(f"- {rule['title']}: {rule['text']}")
n_pass = int(table["passes_all"].sum())
print(f"\n{n_pass:,} of {len(table):,} compositions pass every rule")
for rule in space["rules"]:
    window = rule_window(rule["params"])
    if window and table[rule["name"]].notna().any():
        label, unit = measured(rule)
        hist_prop(table[rule["name"]], label, unit, colour=BLOCK["rules"], window=window, title=rule["title"],
                  ylabel="compositions")
funnel_plot(funnel_stages(table, space["rules"]), colour=BLOCK["rules"])

**Try it:** change the tolerance-factor window and see how many compositions remain. Tick *use_for_search* to keep your window for the search; it is stored in `meidnet.yaml` under `generation.overrides`.

In [ ]:
#@title Change the tolerance-factor window
t_min = 0.80           #@param {type:"slider", min:0.6, max:1.0, step:0.01}
t_max = 1.00           #@param {type:"slider", min:0.9, max:1.2, step:0.01}
use_for_search = True  #@param {type:"boolean"}

In [ ]:
default = fam.constraint_params("tolerance_factor")
fam_search, space_search = fam, space
if default is None:
    print("this family has no tolerance-factor rule")
else:
    cfg.generation.overrides = {"tolerance_factor": {"min": t_min, "max": t_max}}
    fam_new = family_for(cfg, need_variant=True)
    table_new = space_table(enumerate_space(fam_new))     # the rules only: no predictions needed to count
    print(f"family window {default['min']} to {default['max']}: {n_pass:,} compositions pass every rule")
    print(f"your window   {t_min} to {t_max}: {int(table_new['passes_all'].sum()):,} compositions pass every rule")
    hist_prop(table_new["tolerance_factor"], "tolerance factor t", colour=BLOCK["rules"], window=(t_min, t_max),
              title="Goldschmidt tolerance factor, your window", ylabel="compositions")
    if use_for_search:
        fam_search, space_search = fam_new, enumerate_space(fam_new, lm)
    else:
        cfg.generation.overrides = {}
table_search = space_table(space_search)
print_flow("Targets", f"{int(table_search['passes_all'].sum()):,} of {len(table_search):,} compositions pass every rule")

<div style="border-left:6px solid #e87ba4;background:rgba(232,123,164,0.09);border-radius:6px;padding:12px 18px;margin:6px 0">
<h2 style="margin:0 0 6px 0">🎯 Block 5 · Targets</h2>
<p style="margin:0">What the material should have, and how much each goal matters.</p>
</div>

Set one target value per property. Properties you leave out aim at the median of your data. The allowed compositions are ranked with the objectives of `meidnet.yaml` (one squared distance per property), the score the search uses for its candidates.

In [ ]:
#@title Your targets
targets_text = ""  #@param {type:"string"}
#@markdown For example `dir_gap=1.5, heat_all=-0.1`. Leave empty to aim every property at the median of your data.

In [ ]:
columns = list(lm.stats.columns)
target = {p.column: round(float(np.median([r.properties[j] for r in records])), 3)
          for j, p in enumerate(cfg.data.properties)}
for part in filter(None, (s.strip() for s in targets_text.replace(";", ",").split(","))):
    name, value = (x.strip() for x in part.split("="))
    if name not in target:
        raise ValueError(f"'{name}' is not one of your properties: {', '.join(target)}")
    target[name] = float(value)
print("target:", target)
ranges = property_ranges(lm)
for col, value in target.items():
    lo, hi = ranges.get(col, (value, value))
    if not lo <= value <= hi:
        print(f"note: {col} = {value} lies outside the training range {lo:.3g} to {hi:.3g}: the model would be extrapolating")
objectives = [o.model_dump() for o in cfg.generation.objectives]
allowed = table_search[table_search["passes_all"]]
if allowed.empty:
    print("No composition passes every rule: widen a window in the Rules block or choose another variant.")
top10 = rank_by_target(allowed, target, objectives).head(10)
sites = [c for c in table_search.columns if c.startswith("site_")]
display(top10[["formula", *sites, *[f"pred_{c}" for c in columns], "score"]].round(3).reset_index(drop=True))
x = f"pred_{columns[0]}"
y = f"pred_{columns[1]}" if len(columns) > 1 else "a"
space_scatter(table_search, x, y, colour=BLOCK["targets"], highlight=top10,
              target=(target[columns[0]], target[columns[1]] if len(columns) > 1 else None),
              xlabel=f"predicted {columns[0]}", ylabel=f"predicted {columns[1]}" if len(columns) > 1 else "cell edge a [Å]")
hist_prop(allowed[x], f"predicted {columns[0]}", lm.stats.units[0], colour=BLOCK["targets"],
          target=target[columns[0]], title=f"Predicted {columns[0]} of the allowed compositions", ylabel="compositions")

The design space in 3D (chemiscope), with your model's predictions: one point per composition, coloured by whether it passes every rule.

In [ ]:
from meidnet.studio.chemiscope import space_dataset
show_dataset(space_dataset(fam_search, space_search, lm))
print_flow("Search", f"target {target}" + (f" (closest allowed composition: {top10['formula'].iloc[0]})" if len(top10) else ""))

<div style="border-left:6px solid #008300;background:rgba(0,131,0,0.09);border-radius:6px;padding:12px 18px;margin:6px 0">
<h2 style="margin:0 0 6px 0">🔎 Block 6 · Search</h2>
<p style="margin:0">The latent optimisation of the paper: from targets back to structures.</p>
</div>

The ranking above only scores compositions of the prototype. The **search** of the paper works the other way round: it starts from the point of the latent space your targets map to, moves through the space by gradient descent, decodes crystals on the way and keeps the ones that pass every rule. Each candidate is printed as soon as it is saved. The budget below is small so that it runs on a CPU in about a minute; raise it for a real study.

In [ ]:
#@title Search budget
n_candidates = 3  #@param {type:"integer"}
rounds = 3        #@param {type:"integer"}
steps = 300       #@param {type:"integer"}
#@markdown A round moves 24 latent points for *steps* gradient steps; the search stops after *rounds* rounds or once *n_candidates* candidates are saved.

In [ ]:
from meidnet.generate import Designer
gen = cfg.generation                          # the generation section of meidnet.yaml, with your choices
gen.targets, gen.per_target, gen.population, gen.rounds, gen.steps = [target], n_candidates, 24, rounds, steps

names = dict(zip(lm.stats.columns, lm.stats.labels))
units = dict(zip(lm.stats.columns, lm.stats.units))

def on_saved(candidate, target_log):          # called by the search for every candidate it saves
    preds = ", ".join(f"{names[k].lower()} {v:.3g} {units[k]}".strip() for k, v in candidate.predictions.items())
    print(f"  found {candidate.formula:<10} {preds}   (round {candidate.round})")

def progress(*parts):                         # the search's own progress lines, without repeating the finds
    line = " ".join(str(p) for p in parts).strip("\n")
    if not line.strip().startswith("saved"):
        print(line)

res = Designer(lm, fam_search, gen, log=progress, on_saved=on_saved).run(os.path.join(cfg.out_dir, "generation"),
                                                                         ranges=property_ranges(lm))
print_flow("Candidates", f"{len(res.saved)} candidate(s), written to {os.path.join(cfg.out_dir, 'generation')}")

<div style="border-left:6px solid #4a3aa7;background:rgba(74,58,167,0.09);border-radius:6px;padding:12px 18px;margin:6px 0">
<h2 style="margin:0 0 6px 0">💎 Block 7 · Candidates</h2>
<p style="margin:0">What came out, and why each one passed.</p>
</div>

Every saved candidate passed every rule. The report explains each one: its elements and cell, the measured value of every rule, and the predicted properties next to your target, plus a *funnel* that shows where the search lost its attempts. Predicted values are the model's estimates: confirm promising candidates with DFT or experiment.

In [ ]:
from meidnet.config import dump_config
from meidnet.report import generation_report
with open("meidnet.yaml", "w", encoding="utf-8") as f:
    f.write(dump_config(cfg))                 # meidnet.yaml now holds exactly the settings of this run
report = generation_report(cfg, lm, fam_search, res, os.path.join(cfg.out_dir, "generation_report.html"))
show_report(report)

The candidates as a table (✓ = the rule passed, with its measured value) and in 3D, from the CIF files the search wrote.

In [ ]:
display(candidates_table(res.saved))
from pymatgen.core import Structure
found = [Structure.from_file(os.path.join(res.out_dir, c.file)) for c in res.saved]
if found:
    show3d(found, properties={"formula": [c.formula for c in res.saved],
                              **{f"pred_{k}": {"values": [c.predictions[k] for c in res.saved], "units": units[k]}
                                 for k in lm.stats.columns},
                              "score": [c.score for c in res.saved]})
else:
    print("No candidate this time: try more rounds, another target, or a wider window in the Rules block.")

**Download everything:** the CIFs, `candidates.csv`, `model.pt`, the three reports and the `meidnet.yaml` that produced them, in one zip file.

In [ ]:
with zipfile.ZipFile("meidnet_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for folder, _, file_names in os.walk(cfg.out_dir):
        for file_name in file_names:
            path = os.path.join(folder, file_name)
            z.write(path, os.path.relpath(path))
    z.write("meidnet.yaml")
print(f"meidnet_results.zip: {os.path.getsize('meidnet_results.zip') / 1e6:.1f} MB")
if IN_COLAB:
    from google.colab import files
    files.download("meidnet_results.zip")
print_flow("Next steps", f"{len(res.saved)} candidate(s), your model and three reports in meidnet_results.zip")

## Where to go next
* Same data, more epochs: raise *epochs* in the Model block (200 is typical) and run again.
* **03**: add your own rule, for example to exclude toxic elements.
* On your computer: `pip install meidnet`, then `meidnet studio meidnet.yaml` opens the seven blocks side by side with your model.
* Rules, element filters and family files: https://babu09-meidnet.hf.space/docs/

<sub>Notebook by Anand Babu. MEIDNet: <i>Multimodal generative AI framework for inverse materials design</i>, npj Computational Materials (2026), <a href="https://doi.org/10.1038/s41524-026-02153-3">doi:10.1038/s41524-026-02153-3</a>. 3D views use chemiscope: G. Fraux, R. K. Cersonsky, M. Ceriotti, <i>Chemiscope: interactive structure-property explorer for materials and molecules</i>, JOSS 5, 2117 (2020).</sub>